In [3]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

In [4]:
import json
from google.colab import files

# --- Upload your original dialogue.json ---
uploaded = files.upload()  # select dialogue.json
with open("dialogue.json") as f:
    original = json.load(f)

print(f"Original dialogues: {len(original)}")

# --- 40 additional dialogues (fixed for reproducibility) ---
EXTRA_TOPICS = [
    ("robotics", "home robots and daily assistance"),
    ("space", "commercial space travel for ordinary people"),
    ("food", "lab-grown meat and food technology"),
    ("music", "AI-generated music and copyright"),
    ("gaming", "the rise of cloud gaming platforms"),
    ("privacy", "personal data protection in the digital age"),
    ("transport", "autonomous vehicles on public roads"),
    ("energy", "nuclear fusion as a future power source"),
    ("aging", "technology for elderly care and independence"),
    ("language", "how translation AI affects language learning"),
    ("design", "generative AI in product design workflows"),
    ("farming", "precision agriculture and drone technology"),
    ("mental_health", "digital therapy and mental wellness apps"),
    ("housing", "3D-printed houses and affordable construction"),
    ("journalism", "AI-assisted reporting and media trust"),
    ("sports", "data analytics changing team strategy"),
    ("law", "AI tools in legal research and contracts"),
    ("retail", "cashier-free stores and shopping automation"),
    ("water", "smart systems for water conservation"),
    ("tourism", "virtual reality travel experiences"),
    ("supply_chain", "AI-driven logistics and delivery optimization"),
    ("cybersecurity", "defending small businesses from cyber attacks"),
    ("fashion", "sustainable fashion and AI trend prediction"),
    ("construction", "building information modeling and AI planning"),
    ("insurance", "automated claims processing with AI"),
    ("archaeology", "satellite imaging and AI in discovering ruins"),
    ("childcare", "screen time management tools for parents"),
    ("volunteering", "platforms connecting volunteers with local causes"),
    ("accessibility", "AI tools that help visually impaired users"),
    ("sleep", "sleep tracking technology and its real impact"),
    ("pets", "wearable health monitors for pets"),
    ("art", "AI-assisted art restoration in museums"),
    ("waste", "smart recycling systems in urban areas"),
    ("reading", "AI-curated reading lists and book recommendations"),
    ("voting", "digital voting systems and election security"),
    ("fitness", "AI personal trainers and workout planning"),
    ("shipping", "autonomous cargo ships and maritime logistics"),
    ("dentistry", "AI diagnostics in dental imaging"),
    ("libraries", "how public libraries adapt to the digital era"),
    ("comedy", "can AI understand and generate humor"),
]

TEMPLATES = [
    [
        ("host", "Today we are diving into {topic_desc}. What should people know first?"),
        ("expert", "The key starting point is understanding what problem this solves. Most people hear the buzzword but miss the practical value."),
        ("host", "So awareness comes before adoption. What does a realistic first step look like?"),
        ("expert", "Start with a small pilot. Test one specific use case, measure the outcome, and then decide whether to expand."),
    ],
    [
        ("host", "Our topic today is {topic_desc}. Where do you see the biggest opportunity?"),
        ("expert", "The biggest opportunity is in reducing repetitive effort. When people spend less time on routine tasks, they can focus on creative work."),
        ("host", "But there must be challenges as well. What holds people back?"),
        ("expert", "Usually it is a mix of cost, trust, and habit. Change is hard even when the benefits are clear."),
    ],
    [
        ("host", "Let us talk about {topic_desc}. Is the hype justified?"),
        ("expert", "Some of it is. The technology works, but the gap between a demo and real-world deployment is still large."),
        ("host", "What would help close that gap faster?"),
        ("expert", "Better standards, more open data, and honest conversations about limitations. Overpromising slows down real progress."),
    ],
    [
        ("host", "We are exploring {topic_desc} in this episode. Who benefits the most?"),
        ("expert", "Right now, early adopters with technical resources benefit first. But the long-term goal should be broad accessibility."),
        ("host", "How do we make sure it reaches everyone, not just a few?"),
        ("expert", "Public investment, open-source tools, and education programs all play a role. No single solution is enough on its own."),
    ],
]

augmented = list(original)
for i, (topic, desc) in enumerate(EXTRA_TOPICS):
    template = TEMPLATES[i % len(TEMPLATES)]
    turns = [{"speaker": sp, "text": txt.format(topic_desc=desc)} for sp, txt in template]
    augmented.append(
        {
            "dialogue_id": f"dlg_{len(original) + i + 1:03d}",
            "topic": topic,
            "turns": turns,
        }
    )

with open("dialogue_50.json", "w") as f:
    json.dump(augmented, f, indent=2)

print(f"Total dialogues: {len(augmented)}")

Saving dialogue.json to dialogue.json
Original dialogues: 10
Total dialogues: 50


In [10]:
import json

with open("dialogue_50.json") as f:
    dialogues = json.load(f)

sft_data = []
for dlg in dialogues:
    messages = [
        {
            "role": "system",
            "content": (
                f"You are an insightful podcast guest. The topic is {dlg['topic']}. "
                "Give concise, thoughtful answers to the host's questions."
            ),
        }
    ]
    for turn in dlg["turns"]:
        role = "user" if turn["speaker"] == "host" else "assistant"
        messages.append({"role": role, "content": turn["text"]})
    sft_data.append({"messages": messages})

with open("train_sft.json", "w") as f:
    json.dump(sft_data, f, indent=2)

print(f"SFT samples: {len(sft_data)}")
print("Example:")
print(json.dumps(sft_data[0], indent=2))

dataset = load_dataset("json", data_files="train_sft.json", split="train")

def to_text(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,  # 训练时不要加生成提示
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(
    to_text,
    batched=True,
    remove_columns=dataset.column_names,  # 只保留 text
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir="./output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        warmup_steps=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
    ),
)

SFT samples: 50
Example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are an insightful podcast guest. The topic is ai. Give concise, thoughtful answers to the host's questions."
    },
    {
      "role": "user",
      "content": "Welcome back to the show. Today we are looking at how AI is changing everyday learning."
    },
    {
      "role": "assistant",
      "content": "The biggest shift is personalization. Students can now get feedback that matches their pace and skill level."
    },
    {
      "role": "user",
      "content": "That sounds helpful, but some people worry that students may rely too much on automation."
    },
    {
      "role": "assistant",
      "content": "That risk is real. The best use of AI is as a support tool, not as a replacement for thinking."
    }
  ]
}


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

In [1]:
!pip install torch torchvision matplotlib
# Check available GPU
!nvidia-smi

Thu Apr  2 16:52:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import torch

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")

if "A100" in gpu_name:
    MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
    BATCH_SIZE = 4
else:
    MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
    BATCH_SIZE = 2

print(f"Using model: {MODEL_NAME}, batch_size: {BATCH_SIZE}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

# ✅ 换成这个
dataset = load_dataset("json", data_files="train_sft.json", split="train")

def to_text(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(to_text, batched=True, remove_columns=dataset.column_names)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",      # 告诉 Trainer 用 "text" 这列
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir="./output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        num_train_epochs=5,
        warmup_steps=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_strategy="epoch",
        optim="adamw_8bit",
        seed=42,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete. Final loss: {stats.training_loss:.4f}")

GPU: Tesla T4
Using model: unsloth/Llama-3.2-1B-Instruct, batch_size: 2
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 5 | Total steps = 35
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
1,4.180585
2,4.206690
3,4.018383
4,3.609664
5,3.336586
6,3.097296
7,2.830094
8,2.548709
9,2.315969
10,2.113472


Training complete. Final loss: 1.7688


In [14]:
# ✅ 修复版推理循环
for topic, question in test_prompts:
    messages = [
        {
            "role": "system",
            "content": (
                f"You are an insightful podcast guest. The topic is {topic}. "
                "Give concise, thoughtful answers to the host's questions."
            ),
        },
        {"role": "user", "content": question},
    ]

    # apply_chat_template 返回字符串，再用 tokenizer 编码成 tensor
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_ids = inputs["input_ids"]

    outputs = model.generate(
        input_ids,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
    )
    # 只解码新生成的部分
    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    print(f"\n[Topic: {topic}]")
    print(f"Host: {question}")
    print(f"Expert: {response}")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn


[Topic: climate]
Host: What is the most overlooked aspect of climate change policy?
Expert: The most overlooked aspect is the long-term care component. Most conversations focus on short-term fixes, but the real challenge is ensuring that solutions are sustained over years, decades, and generations.
------------------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Topic: ai]
Host: How should universities teach AI ethics to engineering students?
Expert: Right first, then. Start with the basics. Teach the why, not just the how. Give students a clear understanding of the problem and the potential solutions. Then, let them explore the practical aspects. But make sure they understand the limitations as well.
------------------------------------------------------------

[Topic: media]
Host: Do you think traditional newspapers will survive another decade?
Expert: For now, yes. But the medium is changing. Digital will continue to grow.
------------------------------------------------------------


In [17]:
from google.colab import files

# 保存 LoRA adapter
model.save_pretrained("podcast-expert-lora")
tokenizer.save_pretrained("podcast-expert-lora")

# 打包下载
!zip -r podcast-expert-lora.zip podcast-expert-lora/
files.download("podcast-expert-lora.zip")

print("Done. Check your Downloads folder.")

  adding: podcast-expert-lora/ (stored 0%)
  adding: podcast-expert-lora/adapter_model.safetensors (deflated 8%)
  adding: podcast-expert-lora/tokenizer_config.json (deflated 45%)
  adding: podcast-expert-lora/tokenizer.json (deflated 85%)
  adding: podcast-expert-lora/README.md (deflated 65%)
  adding: podcast-expert-lora/adapter_config.json (deflated 58%)
  adding: podcast-expert-lora/chat_template.jinja (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done. Check your Downloads folder.
